# Clase 15: Caminos mínimos y Dijkstra

## Pregunta inicial

¿Cómo encontramos el camino de menor costo cuando las aristas no cuestan lo mismo?

> [!IMPORTANT]
> No respondas dentro del notebook. Responde en `notebook.md`, siguiendo el mismo orden.

**Responde esta pregunta en notebook.md.**


## Objetivos

Al final de la clase deberías poder:

- explicar por qué BFS no basta con pesos diferentes;
- distinguir distancia por aristas y distancia por costo;
- mantener distancias tentativas y predecesores;
- ejecutar y explicar una relajación;
- usar una cola de prioridad de pares `(distancia, nodo)`;
- ejecutar Dijkstra manualmente;
- detectar entradas obsoletas;
- reconstruir un camino mínimo;
- analizar complejidad y limitaciones;
- implementar y probar Dijkstra con pesos no negativos.


## 1. Por qué BFS ya no basta

En una red de ciudades, una carretera directa no siempre es la ruta más barata. Considera:

```text
A --10--> D
 \         ▲
  1        | 1
   ▼       |
    B -----+
```

El camino `A → D` tiene una arista y costo 10. El camino `A → B → D` tiene dos aristas, pero costo total 2. BFS procesa por niveles: descubre D en el primer nivel y B también en el primero; su garantía de “camino más corto” cuenta aristas, no suma pesos diferentes.

En grafos no ponderados, distancia significa número de aristas. En grafos ponderados, distancia significa suma de los pesos del camino. La palabra “corto” depende entonces del modelo: tiempo, dinero, longitud o consumo.

> [!IMPORTANT]
> Cuando las aristas tienen costos diferentes, necesitamos procesar primero la menor distancia acumulada conocida, no necesariamente el nodo con menos aristas desde el origen.

**Pregunta adicional:** ¿Qué camino elegiría una estrategia que solo compara niveles y cuál debería elegir si optimizamos costo total?

**Responde esta pregunta en notebook.md.**

Ya encontramos la limitación; recordemos exactamente qué garantía sí ofrece BFS.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué contar aristas ya no es suficiente en un grafo ponderado?

**Responde esta pregunta en notebook.md.**


## 2. Recordatorio de BFS

BFS usa una cola FIFO y procesa todos los nodos a una arista del origen antes que los de dos aristas. Esa estrategia es correcta para caminos mínimos cuando cada arista tiene costo unitario o, de forma equivalente, cuando todos los pesos son iguales.

| Característica | BFS | Dijkstra |
| --- | --- | --- |
| Tipo de grafo | no ponderado o pesos iguales | pesos no negativos |
| Prioridad de procesamiento | nivel más cercano | menor costo acumulado |
| Estructura auxiliar | cola FIFO | cola de prioridad |
| Resultado | menos aristas | menor costo total |

No repetiremos toda la Clase 10. Conservaremos una idea: la estructura de pendientes determina qué elemento se procesa después. En BFS, la llegada a la cola representa el orden por niveles. Con pesos distintos, necesitamos que una candidatura posterior pueda adelantarse si su costo es menor.

> [!NOTE]
> Dijkstra no “mejora BFS” para todo tipo de grafo. Resuelve otro contrato: pesos no negativos y costo acumulado.

Ahora construiremos una red que reutilizaremos para descubrir las piezas del nuevo algoritmo.

### Comprueba tu comprensión

**Pregunta:** ¿En qué condición BFS sí garantiza caminos de costo mínimo?

**Responde esta pregunta en notebook.md.**


## 3. Red de ciudades conductora

Usaremos la misma red durante la tabla manual, la relajación, el heap, el visualizador y la reconstrucción:

```text
        4           1
   A ------> B ----------> D
    \        |             |
   1\       7|            3|
      ▼      ▼             ▼
       C ----+-----------> E
        \ 2  ▲      5
         └──>B   C ------> D
```

Lista de adyacencia:

```python
{
    "A": [("B", 4), ("C", 1)],
    "B": [("D", 1), ("E", 7)],
    "C": [("B", 2), ("D", 5)],
    "D": [("E", 3)],
    "E": [],
}
```

Cada nodo representa una ciudad; una arista dirigida, una carretera disponible; el peso, tiempo de viaje. Buscamos desde A el costo mínimo a todas las ciudades. La dirección importa: que exista `C → B` no implica `B → C`.

**Actividad manual:** enumera al menos dos caminos de A a E y suma sus pesos. No decidas todavía cuál usará el algoritmo.

**Responde esta pregunta en notebook.md.**

Para comparar rutas sin enumerarlas todas, mantendremos una mejor distancia conocida por ciudad.

### Comprueba tu comprensión

**Pregunta:** ¿Qué representan nodos, aristas y pesos en esta red?

**Responde esta pregunta en notebook.md.**


## 4. Distancias tentativas

Al iniciar solo conocemos un camino de costo cero al origen. Para los demás nodos usamos infinito, que significa “todavía no conocemos ningún camino”, no que exista una carretera de peso infinito.

| Nodo | Distancia conocida | Predecesor |
| --- | ---: | --- |
| A | 0 | — |
| B | ∞ | — |
| C | ∞ | — |
| D | ∞ | — |
| E | ∞ | — |

La distancia es **tentativa** mientras puede aparecer otra ruta más barata. El predecesor registra desde qué nodo llegó la mejor ruta conocida y permitirá reconstruir el camino al final.

Después de examinar A, conoceremos candidaturas para B y C. Que B tenga distancia 4 no significa todavía que 4 sea la mejor posible: el camino A→C→B podría mejorarla. Una distancia extraída con el menor valor pendiente puede tratarse como estable bajo la condición de pesos no negativos; explicaremos esa condición antes de usar la palabra “definitiva”.

> [!TIP]
> En papel, actualiza distancia y predecesor en la misma fila. Una distancia sin su origen pierde información para reconstruir la ruta.

Ya tenemos memoria del mejor costo; ahora necesitamos una operación que intente mejorarlo arista por arista.

### Comprueba tu comprensión

**Pregunta:** ¿Qué significa infinito en la tabla de distancias?

**Responde esta pregunta en notebook.md.**


## 5. Relajación

Relajar una arista significa **intentar mejorar la mejor distancia conocida usando esa arista**. Si conocemos la distancia a `u` y existe `u → v` con peso `w`, calculamos:

\(\text{candidata} = \text{distancia}[u] + w\)

Si la candidata es menor que `distancia[v]`, actualizamos distancia y predecesor.

Ejemplo: distancia a B = 4, arista B→D de peso 2 y distancia actual a D = 10. La candidata es \(4 + 2 = 6\). Como \(6 < 10\), D mejora a 6 y su predecesor pasa a B.

Cuatro casos:

| Actual | Arista | Anterior | Candidata | Decisión |
| ---: | --- | ---: | ---: | --- |
| 4 | B→D, 2 | 10 | 6 | actualizar |
| 4 | B→D, 8 | 10 | 12 | conservar |
| 0 | A→C, 1 | ∞ | 1 | actualizar |
| 1 | C→B, 2 | 4 | 3 | volver a mejorar B |

Relajar no recorre físicamente el camino, no cambia el peso y no acepta para siempre una ruta en ese instante.

Ahora observaremos cómo una misma distancia puede mejorar varias veces.

### Comprueba tu comprensión

**Pregunta:** ¿Qué tres valores comparamos al relajar una arista?

**Responde esta pregunta en notebook.md.**


## 6. Mejoras sucesivas

En la red conductora, B se descubre desde A con costo 4. Después procesamos C con costo 1 y relajamos C→B de peso 2: aparece una candidatura 3. B mejora de 4 a 3 y el predecesor cambia de A a C.

Podemos insertar `(3, B)` en el heap aunque `(4, B)` siga presente. Borrar una entrada arbitraria dentro de un heap complicaría la implementación. En lugar de eso usamos **eliminación perezosa**: cuando salga `(4, B)`, la comparamos con `distancia[B]`, que ya vale 3. Como no coinciden, la entrada es obsoleta y se ignora.

```text
heap: (3,B), (4,B), (6,D)
distancia[B] = 3

extraer (3,B) → vigente, relajar vecinos
extraer (4,B) → obsoleta, continuar
```

> [!IMPORTANT]
> Una entrada obsoleta no significa que el heap esté roto. Es una candidatura histórica que perdió frente a una mejor.

La actualización genera varias candidaturas pendientes. Necesitamos decidir cuál procesar primero.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué actualizar una distancia no obliga a borrar inmediatamente su entrada anterior del heap?

**Responde esta pregunta en notebook.md.**


## 7. Descubrimiento de la cola de prioridad

Después de varias relajaciones podríamos tener:

```text
(3, C)
(7, B)
(10, D)
```

¿Cuál nodo pendiente debemos procesar? El de menor distancia conocida: `(3, C)`. La Clase 14 nos dio exactamente esta operación. En el par `(distancia, nodo)`, la distancia es la prioridad y el nodo es el elemento. Menor distancia significa mayor prioridad de procesamiento.

Una cola FIFO conservaría el orden de descubrimiento y podría procesar primero una ruta costosa. El min-heap reordena automáticamente las candidaturas por distancia. Consultar el mínimo cuesta O(1) y extraerlo O(log n).

En Python, `heapq` ordena tuplas lexicográficamente: primero distancia y, si hay empate, nodo. El desempate no afecta el costo mínimo; puede cambiar cuál camino igualmente óptimo queda registrado como predecesor.

> [!NOTE]
> La cola contiene candidaturas, no una lista de distancias ya terminadas.

Ya construimos la necesidad completa; ahora podemos nombrar formalmente el algoritmo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué componente del par funciona como prioridad y cuál como elemento?

**Responde esta pregunta en notebook.md.**


## 8. Algoritmo de Dijkstra

El procedimiento descubierto se llama **algoritmo de Dijkstra**, por Edsger W. Dijkstra, científico de la computación neerlandés que lo formuló en la década de 1950. La historia importa menos que la idea: calcular caminos mínimos desde un origen en un grafo con pesos no negativos.

El algoritmo devuelve dos productos:

- `distancias[nodo]`: costo mínimo conocido desde el origen;
- `predecesores[nodo]`: nodo anterior en un camino mínimo.

Los pesos no negativos son esenciales. Cuando extraemos la menor distancia pendiente, ninguna ruta futura que pase por aristas de costo ≥ 0 puede regresar con un costo menor. Una arista negativa rompería ese argumento: un nodo procesado podría mejorar más tarde a través de un costo que reduce la suma.

No exigimos que el grafo sea conectado. Los nodos fuera de la componente alcanzable conservan infinito y predecesor `None`.

> [!WARNING]
> Dijkstra no debe usarse silenciosamente con pesos negativos. La implementación de esta clase los rechazará con `ValueError`.

Con el contrato claro, reunamos las operaciones en pseudocódigo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué problema resuelve Dijkstra y qué restricción tienen los pesos?

**Responde esta pregunta en notebook.md.**


## 9. Pseudocódigo y entradas obsoletas

```text
para cada nodo:
    distancia[nodo] = infinito
    predecesor[nodo] = ninguno

distancia[origen] = 0
insertar (0, origen) en la cola de prioridad

mientras la cola no esté vacía:
    extraer (distancia_extraida, actual)

    si distancia_extraida no coincide con distancia[actual]:
        ignorar esta entrada y continuar

    para cada (vecino, peso) de actual:
        candidata = distancia[actual] + peso
        si candidata < distancia[vecino]:
            distancia[vecino] = candidata
            predecesor[vecino] = actual
            insertar (candidata, vecino)
```

El orden de las comprobaciones importa. Primero extraemos el mínimo, después descartamos una posible entrada obsoleta y solo entonces recorremos aristas salientes. Si relajáramos desde una entrada vieja, haríamos trabajo innecesario y complicaríamos el razonamiento.

**Actividad:** subraya en el pseudocódigo las tres operaciones que dependen directamente de la Clase 14.

**Responde esta pregunta en notebook.md.**

Ya tenemos la receta; la entenderemos mejor ejecutándola a mano.

### Comprueba tu comprensión

**Pregunta:** ¿En qué momento se ignora una entrada obsoleta?

**Responde esta pregunta en notebook.md.**


## 10. Recorrido manual mínimo

Grafo: A→B con 1, A→C con 4 y B→C con 2.

| Paso | Extraído | Relajación | Distancias A,B,C | Heap después |
| ---: | --- | --- | --- | --- |
| 0 | — | inicializar | 0,∞,∞ | `(0,A)` |
| 1 | `(0,A)` | B=1, C=4 | 0,1,4 | `(1,B),(4,C)` |
| 2 | `(1,B)` | C mejora a 3 | 0,1,3 | `(3,C),(4,C)` |
| 3 | `(3,C)` | sin vecinos | 0,1,3 | `(4,C)` |
| 4 | `(4,C)` | obsoleta | 0,1,3 | vacío |

Observa que C aparece dos veces. Solo la candidatura 3 es vigente. El camino a C se reconstruye con predecesores: C←B←A y luego se invierte.

> [!TIP]
> En una traza manual no borres la entrada vieja; márcala y demuestra cuándo se descarta.

Ahora aplicaremos el mismo patrón a la red conductora con cinco ciudades.

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es el orden de extracción vigente en el ejemplo mínimo?

**Responde esta pregunta en notebook.md.**


## 11. Recorrido manual intermedio

Completa la tabla antes de usar el visualizador:

| Extracción vigente | Mejoras producidas | Distancias después | Predecesores que cambian |
| --- | --- | --- | --- |
| `(0,A)` | B=4, C=1 | A0 B4 C1 D∞ E∞ | B←A, C←A |
| `(1,C)` | B=3, D=6 | A0 B3 C1 D6 E∞ | B←C, D←C |
| `(3,B)` | | | |
| `(4,D)` | | | |
| `(7,E)` | | | |

Además aparecerán entradas obsoletas `(4,B)`, `(6,D)` y `(10,E)`. Completa qué comparación permite ignorarlas.

La distancia de D mejora de 6 a 4 cuando se procesa B. E se descubre con 10 desde B y mejora a 7 desde D. Esta secuencia muestra que “descubierto” no significa “terminado”.

**Responde esta pregunta en notebook.md.**

La tabla produce distancias; ahora aprenderemos a convertir predecesores en una ruta legible.

### Comprueba tu comprensión

**Pregunta:** ¿Cuáles son las distancias finales desde A en la red conductora?

**Responde esta pregunta en notebook.md.**


## 12. Reconstrucción del camino

El mapa de predecesores apunta hacia atrás: `predecesor[E]=D`, `predecesor[D]=B`, `predecesor[B]=C` y `predecesor[C]=A`. Empezamos en E:

```text
E ← D ← B ← C ← A
```

La secuencia se obtuvo de destino a origen, así que la invertimos:

```text
A → C → B → D → E
```

Si llegamos a `None` antes de alcanzar el origen, el destino es inalcanzable y devolvemos `[]`. Para origen igual a destino, el camino es `[origen]`. Una implementación robusta también evita ciclos accidentales en un mapa de predecesores inválido.

> [!NOTE]
> Dijkstra calcula todas las distancias desde el origen. Reconstruir un destino concreto es una operación posterior sobre el mapa producido.

Ya sabemos leer el resultado; el visualizador permitirá inspeccionar cómo se construyó.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué recorremos predecesores desde el destino y después invertimos?

**Responde esta pregunta en notebook.md.**


## 13. Visualización interactiva

El visualizador presenta simultáneamente la red ponderada, la tabla de distancias/predecesores y el min-heap de candidaturas. Azul marca el nodo actual, naranja la arista relajada, verde el vecino candidato y rojo una entrada obsoleta. Las etiquetas acompañan al color.

Usa Anterior y Siguiente para justificar cada comparación. Reproducir y Pausar recorren la secuencia sin `time.sleep`; el slider permite saltar. En Modo reto se oculta la decisión para elegir entre actualizar, conservar, relajar o ignorar una entrada vieja.

Antes de avanzar observa:

- distancia extraída y distancia vigente;
- peso de la arista activa;
- candidata calculada;
- par mínimo del heap;
- predecesor antes y después.

**Actividad:** en “Entradas obsoletas”, pausa antes de extraer `(10,B)` y predice la decisión.

**Responde esta pregunta en notebook.md.**

El visualizador explica estados ya preparados; la implementación evaluada seguirá viviendo en la carpeta de entrega.

### Comprueba tu comprensión

**Pregunta:** ¿Qué tres representaciones deben permanecer sincronizadas en cada paso?

**Responde esta pregunta en notebook.md.**


In [ ]:
import ipywidgets as widgets
widgets.IntSlider(description="Prueba")


In [ ]:
from pathlib import Path
import sys
candidatas=[Path.cwd(),Path.cwd().parent,Path.cwd()/'clase_15',Path.cwd().parent/'clase_15']
RAIZ_CLASE=next((r for r in candidatas if (r/'src'/'visualizador_dijkstra.py').exists()),None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('Abre desde curso-alumnos o clase_15/notebooks')
sys.path.insert(0,str(RAIZ_CLASE))
from src.visualizador_dijkstra import diagnosticar_widgets, mostrar_dijkstra
print(diagnosticar_widgets())
mostrar_dijkstra()


## 14. Implementación

La entrega separa tres responsabilidades:

```python
def dijkstra(grafo, origen) -> tuple[dict[str, float], dict[str, str | None]]:
    ...

def reconstruir_camino(predecesores, origen, destino) -> list[str]:
    ...

def camino_minimo(grafo, origen, destino) -> tuple[float, list[str]]:
    ...
```

`dijkstra` valida pesos, calcula distancias y registra predecesores. `reconstruir_camino` no vuelve a recorrer el grafo: sigue el mapa. `camino_minimo` coordina ambas y devuelve costo/camino para un destino.

Contratos importantes: no mutar el grafo, rechazar pesos negativos, usar infinito para inalcanzables, devolver camino vacío para destino inalcanzable y lanzar `KeyError` si origen o destino no pertenece al grafo.

> [!CAUTION]
> No copies la lógica del visualizador: consume estados precomputados y no es la solución de la práctica.

Después de implementar, justificaremos cuánto trabajo realiza.

### Comprueba tu comprensión

**Pregunta:** ¿Qué responsabilidad tiene cada una de las tres funciones de la entrega?

**Responde esta pregunta en notebook.md.**


## 15. Complejidad

Sea \(V\) la cantidad de nodos y \(E\) la cantidad de aristas. Inicializar diccionarios cuesta O(V). Cada arista se examina al relajar desde su origen: O(E). Una mejora inserta una candidatura y cada extracción usa el heap, con costo O(log V) de manera simplificada.

La complejidad habitual con lista de adyacencia y min-heap se expresa como:

\(O((V + E)\log V)\)

En grafos conectados suele abreviarse O(E log V). Las entradas obsoletas pueden aumentar extracciones, pero cada una proviene de una mejora previa y mantiene el mismo orden asintótico.

El espacio es O(V+E) para grafo, distancias, predecesores y candidaturas. Reconstruir un camino cuesta O(k), donde k es su número de nodos.

Comparación: buscar linealmente el menor nodo pendiente en cada iteración lleva a O(V²), opción que puede ser aceptable en grafos densos pequeños, pero pierde la conexión con la cola de prioridad de Clase 14.

Ya conocemos el costo; aplicaremos el algoritmo a una situación guiada completa.

### Comprueba tu comprensión

**Pregunta:** ¿De dónde proviene el factor logarítmico de Dijkstra con heap?

**Responde esta pregunta en notebook.md.**


## 16. Problema guiado: entrega urgente

Una empresa debe enviar un paquete desde A hasta E. Cada peso representa minutos y todas las carreteras son dirigidas. Resuelve con este flujo:

1. Modela la lista de adyacencia.
2. Inicializa A=0 y el resto en infinito.
3. Registra cada extracción vigente.
4. Para cada arista escribe candidata, comparación y decisión.
5. Marca entradas obsoletas.
6. Reconstruye E siguiendo predecesores.

Tabla de evidencia:

| Concepto | Resultado que debes completar |
| --- | --- |
| Orden de extracciones vigentes | |
| Mejoras de B | |
| Mejoras de D | |
| Mejoras de E | |
| Predecesores finales | |
| Camino A→E | |
| Costo total | |

Compara al final con dos alternativas: A→B→E y A→C→D→E. Explica por qué una ruta con más aristas puede ser mejor.

**Responde esta pregunta en notebook.md.**

> [!TIP]
> Usa el visualizador para comprobar tu traza después de hacerla, no para sustituir el razonamiento manual.

El problema guiado usa pesos válidos; ahora estudiaremos la frontera de aplicación.

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es el costo y camino mínimo de A a E en la red conductora?

**Responde esta pregunta en notebook.md.**


## 17. Limitaciones y pesos negativos

Dijkstra confía en que, una vez extraída la menor distancia vigente, avanzar por nuevas aristas no reducirá el costo acumulado por debajo de ella. Con pesos no negativos, toda extensión mantiene o aumenta la suma.

Si existe una arista negativa, un camino que parece caro puede bajar después. Ejemplo: A→B cuesta 2, A→C cuesta 5 y C→B cuesta -10. Procesar B con 2 sería prematuro porque A→C→B cuesta -5.

La práctica rechazará cualquier peso negativo. Esto es mejor que producir silenciosamente un resultado incorrecto. Para grafos con pesos negativos existen otros algoritmos, como Bellman–Ford, pero no los implementaremos en esta clase.

También distinguimos: aristas de peso cero sí son válidas; nodos inalcanzables conservan infinito; múltiples caminos óptimos pueden producir predecesores distintos con el mismo costo.

> [!WARNING]
> “Ponderado” no implica automáticamente que Dijkstra sea aplicable: revisa el signo de los pesos.

Con los límites claros, las pruebas deben proteger tanto resultados como errores esperados.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué una arista negativa rompe la decisión codiciosa de Dijkstra?

**Responde esta pregunta en notebook.md.**


## 18. Pruebas y revisión técnica

Las pruebas públicas cubren origen, relajación, reconstrucción, inalcanzables, pesos cero, negativos, nodos implícitos y no mutación. Cada estudiante agrega al menos tres: mejora múltiple, destino inalcanzable y ruta directa costosa reemplazada por una indirecta.

Una revisión técnica debe cambiar a la rama del compañero, inspeccionar el diff, ejecutar pruebas públicas y propias, copiar la salida completa de pytest y escribir observaciones accionables. Ejemplo: “En A→B=10, A→C=1, C→B=2, la entrada `(10,B)` relaja vecinos aunque `distancia[B]=3`; agrega la comprobación inmediatamente después de heappop”.

Usa `NOTE` para contexto, `TIP` para estrategia, `IMPORTANT` para contratos, `WARNING` para errores probables y `CAUTION` para riesgos que obligan a detenerse.

**Pregunta adicional:** ¿Qué afirmaciones comprobarías además del costo mínimo para validar la reconstrucción?

**Responde esta pregunta en notebook.md.**

Ya podemos implementar, probar y revisar; cerremos conectando el recorrido completo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué caso de prueba demuestra que manejamos entradas obsoletas correctamente?

**Responde esta pregunta en notebook.md.**


## 19. Práctica adicional

Cuando termines, busca problemas que pidan caminos mínimos desde un origen con pesos no negativos. Variantes útiles:

| Problema | Idea dominante | Nivel |
| --- | --- | --- |
| tiempos de viaje desde una ciudad | menor distancia pendiente | práctica sugerida |
| retraso de una señal en una red | máximo entre distancias mínimas | práctica sugerida |
| cuadrícula con costos de entrada | modelar celdas como nodos ponderados | opcional |
| camino con descuento | ampliar el estado, no usar Dijkstra básico sin cambios | reto |

Antes de programar, responde: ¿los pesos son no negativos?, ¿se pide desde un origen?, ¿necesito una distancia o el camino?, ¿hay estados adicionales? No todo problema con palabras “ruta” y “costo” coincide con el algoritmo básico.

> [!TIP]
> Modelar correctamente nodos, aristas y estado suele ser más difícil que escribir el bucle del heap.

La práctica opcional amplía contexto; la síntesis final conserva la idea esencial.

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación dominante indica que un problema puede resolverse con Dijkstra?

**Responde esta pregunta en notebook.md.**


## 20. Cierre

La clase comenzó con una contradicción para BFS: menos aristas no siempre significa menor costo. Introdujimos distancias tentativas para recordar la mejor ruta conocida y relajación para intentar mejorarla con una arista. Al aparecer varias candidaturas, recuperamos la cola de prioridad: la distancia se convirtió en prioridad y el nodo en elemento.

| Pregunta | Respuesta de la clase |
| --- | --- |
| ¿Qué minimizamos? | suma de pesos |
| ¿Qué guardamos por nodo? | distancia y predecesor |
| ¿Cómo intentamos mejorar? | relajación |
| ¿Qué procesamos después? | menor distancia pendiente |
| ¿Qué estructura lo permite? | min-heap |
| ¿Qué algoritmo surge? | Dijkstra |
| ¿Qué restricción es esencial? | pesos no negativos |

Los predecesores reconstruyen caminos, las entradas obsoletas simplifican el uso de `heapq` y la complejidad queda en O((V+E) log V). La misma lección atraviesa el curso: elegimos la estructura auxiliar según la operación que debe dominar la ejecución.

> [!IMPORTANT]
> Dijkstra no visita simplemente “el nodo más cercano”: procesa la menor distancia acumulada conocida y mejora rutas mediante relajación.

**Responde esta pregunta en notebook.md.**

### Comprueba tu comprensión

**Pregunta:** ¿Qué cadena de decisiones transforma el problema ponderado en Dijkstra?

**Responde esta pregunta en notebook.md.**
